# IMPORT LIBRARIES

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any, Tuple, List, Optional
import json
import math
import os
import random

In [ ]:
def _euclid(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


def _rand_point(rng: random.Random, lo: float, hi: float) -> Tuple[float, float]:
    return (rng.uniform(lo, hi), rng.uniform(lo, hi))


def _sample_without_replacement(rng: random.Random, items: List[str], k: int) -> List[str]:
    k = min(k, len(items))
    items2 = items[:]
    rng.shuffle(items2)
    return items2[:k]


def save_instance_json(data: Dict[str, Any], path: str) -> None:
    """Save instance to JSON. Converts tuple keys to strings automatically."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def encode_keys(obj):
        if isinstance(obj, dict):
            out = {}
            for k, v in obj.items():
                if isinstance(k, tuple):
                    kk = "|".join(map(str, k))
                else:
                    kk = str(k)
                out[kk] = encode_keys(v)
            return out
        if isinstance(obj, list):
            return [encode_keys(x) for x in obj]
        return obj

    with open(path, "w", encoding="utf-8") as f:
        json.dump(encode_keys(data), f, indent=2, ensure_ascii=False)

# -------------------------
# Styles / config
# -------------------------

@dataclass(frozen=True)
class GenConfig:
    style: str = "clustered"        # "clustered" | "scattered" | "multi_region"
    seed: int = 1
    H: int = 30

    n_bases: int = 2
    n_techs: int = 6
    n_plants: int = 20
    n_tickets: int = 40

    modes: Tuple[str, ...] = ("car", "train", "air")

    # Geometry / travel
    coord_lo: float = 0.0
    coord_hi: float = 100.0
    km_per_day_by_mode: Dict[str, float] = None  # if None, defaults below

    # Durations
    dur_min: int = 1
    dur_max: int = 4
    skill_spread: float = 0.35      # variability per technician (higher -> more heterogeneity)
    bigM_dur: int = 999             # if technician cannot do ticket, set dur[t,k]=bigM_dur

    # Tickets timing
    release_window: Tuple[int, int] = (1, 20)  # a_t in [..]
    sla_slack_window: Tuple[int, int] = (2, 10)  # b_t = a_t + slack, clipped to H

    # Hotel rule
    hotel_distance_threshold: float = 40.0      # km from base to plant => hotel required
    hotel_prob_extra: float = 0.10             # additional random chance (noise) for hotel flag

    # Feasibility / capability
    capable_k_per_ticket: Tuple[int, int] = (2, 4)  # each ticket doable by this many techs (approx)

    # Objective weights
    alpha: float = 1.0
    beta: float = 1.0
    eta: float = 1.0
    gamma: float = 10.0

    # Costs
    c_hotel: float = 80.0
    EF_by_mode: Dict[str, float] = None         # kgCO2/km

    # Away policy
    X: int = 5


def _default_km_per_day():
    return {"car": 500.0, "train": 800.0, "air": 1500.0}


def _default_EF():
    # toy numbers; replace with your own assumptions
    return {"car": 0.200, "train": 0.035, "air": 0.285}

# GENERATE DATA

In [3]:
def generate_instance(
    *,
    style: str = "clustered",
    seed: int = 1,
    n_bases: int = 2,
    n_techs: int = 6,
    n_plants: int = 20,
    n_tickets: int = 40,
    H: int = 30,
    X: int = 5,
    modes: Tuple[str, ...] = ("car", "train", "air"),
) -> Dict[str, Any]:
    """
    Generate a synthetic instance and return a `data` dict compatible with build_model(data).
    """

    cfg = GenConfig(
        style=style,
        seed=seed,
        H=H,
        n_bases=n_bases,
        n_techs=n_techs,
        n_plants=n_plants,
        n_tickets=n_tickets,
        modes=modes,
        X=X,
        km_per_day_by_mode=_default_km_per_day(),
        EF_by_mode=_default_EF(),
    )

    rng = random.Random(cfg.seed)

    # -------------------------
    # Create IDs
    # -------------------------
    B = [f"b{b+1}" for b in range(cfg.n_bases)]
    K = [f"k{k+1}" for k in range(cfg.n_techs)]
    P = [f"p{p+1}" for p in range(cfg.n_plants)]
    T = [f"t{t+1}" for t in range(cfg.n_tickets)]
    D = list(range(1, cfg.H + 1))
    M = list(cfg.modes)
    N = B + P

    # -------------------------
    # Coordinates by style
    # -------------------------
    coords: Dict[str, Tuple[float, float]] = {}

    if cfg.style == "clustered":
        # Plants in a few clusters; bases near cluster centers
        n_clusters = max(2, min(5, cfg.n_bases + 1))
        centers = [_rand_point(rng, cfg.coord_lo + 10, cfg.coord_hi - 10) for _ in range(n_clusters)]

        # Bases: pick near distinct centers
        for i, b in enumerate(B):
            c = centers[i % n_clusters]
            coords[b] = (c[0] + rng.uniform(-5, 5), c[1] + rng.uniform(-5, 5))

        # Plants: assign to random centers
        for p in P:
            c = centers[rng.randrange(n_clusters)]
            coords[p] = (c[0] + rng.uniform(-12, 12), c[1] + rng.uniform(-12, 12))

    elif cfg.style == "scattered":
        # Everything uniformly scattered
        for b in B:
            coords[b] = _rand_point(rng, cfg.coord_lo, cfg.coord_hi)
        for p in P:
            coords[p] = _rand_point(rng, cfg.coord_lo, cfg.coord_hi)

    elif cfg.style == "multi_region":
        # Two or three regions far apart; bases anchored in regions; plants distributed by region
        n_regions = 3 if cfg.n_bases >= 3 else 2
        region_centers = [(20, 20), (80, 80), (20, 80)]
        region_centers = region_centers[:n_regions]

        for i, b in enumerate(B):
            c = region_centers[i % n_regions]
            coords[b] = (c[0] + rng.uniform(-6, 6), c[1] + rng.uniform(-6, 6))

        for p in P:
            c = region_centers[rng.randrange(n_regions)]
            coords[p] = (c[0] + rng.uniform(-18, 18), c[1] + rng.uniform(-18, 18))

    else:
        raise ValueError(f"Unknown style='{cfg.style}'. Use: clustered | scattered | multi_region")

    # -------------------------
    # Assign base to technician
    # -------------------------
    base: Dict[str, str] = {}
    for idx, k in enumerate(K):
        base[k] = B[idx % len(B)]  # balanced allocation

    # -------------------------
    # Ticket nodes (plants)
    # -------------------------
    node: Dict[str, str] = {}
    for t in T:
        node[t] = P[rng.randrange(len(P))]

    # -------------------------
    # Ticket timing / SLA
    # -------------------------
    a: Dict[str, int] = {}
    b: Dict[str, int] = {}
    w: Dict[str, float] = {}

    rel_lo, rel_hi = cfg.release_window
    slack_lo, slack_hi = cfg.sla_slack_window

    rel_hi = min(rel_hi, cfg.H)
    for t in T:
        a_t = rng.randint(rel_lo, rel_hi)
        slack = rng.randint(slack_lo, slack_hi)
        b_t = min(cfg.H, a_t + slack)
        a[t] = a_t
        b[t] = b_t
        # weights: mix of 1..5
        w[t] = float(rng.choice([1, 1, 2, 2, 3, 4, 5]))

    # -------------------------
    # Distance + travel time
    # -------------------------
    dist: Dict[Tuple[str, str], float] = {}
    tt: Dict[Tuple[str, str, str], int] = {}

    for i in N:
        for j in N:
            dij = 0.0 if i == j else _euclid(coords[i], coords[j])
            dist[(i, j)] = float(round(dij, 3))
            for m in M:
                km_per_day = cfg.km_per_day_by_mode[m]
                # at least 0; if different node, at least 1 day
                days = 0 if i == j else max(1, int(math.ceil(dij / km_per_day)))
                tt[(i, j, m)] = int(days)

    EF = dict(cfg.EF_by_mode)

    # -------------------------
    # Hotel flags Hkp: plant requires hotel if far from technician base (+ noise)
    # -------------------------
    Hkp: Dict[Tuple[str, str], int] = {}
    for k in K:
        bk = base[k]
        for p in P:
            need = 1 if dist[(bk, p)] > cfg.hotel_distance_threshold else 0
            if rng.random() < cfg.hotel_prob_extra:
                need = 1
            Hkp[(k, p)] = int(need)

    # -------------------------
    # Durations dur[t,k]: known exact time per tech & ticket
    # Also impose capability structure: each ticket doable by ~capable_k_per_ticket techs.
    # For non-capable, set dur to bigM_dur (model will avoid).
    # -------------------------
    dur: Dict[Tuple[str, str], int] = {}
    cap_lo, cap_hi = cfg.capable_k_per_ticket

    # Technician "speed factor"
    tech_factor: Dict[str, float] = {}
    for k in K:
        # around 1.0 with spread
        tech_factor[k] = max(0.6, rng.gauss(1.0, cfg.skill_spread))

    for t in T:
        base_duration = rng.randint(cfg.dur_min, cfg.dur_max)
        # choose who can do it
        cap_k = rng.randint(cap_lo, min(cap_hi, len(K)))
        capable_techs = set(_sample_without_replacement(rng, K, cap_k))

        for k in K:
            if k not in capable_techs:
                dur[(t, k)] = cfg.bigM_dur
            else:
                # exact but heterogeneous durations
                dtk = int(max(1, round(base_duration * tech_factor[k])))
                dur[(t, k)] = dtk

    # -------------------------
    # Pack data dict compatible with build_model()
    # -------------------------
    data: Dict[str, Any] = {
        "B": B,
        "K": K,
        "P": P,
        "T": T,
        "D": D,
        "M": M,
        "N": N,
        "a": a,
        "b": b,
        "w": w,
        "dur": dur,
        "node": node,
        "base": base,
        "dist": dist,
        "tt": tt,
        "EF": EF,
        "Hkp": Hkp,
        "c_hotel": cfg.c_hotel,
        "X": cfg.X,
        "alpha": cfg.alpha,
        "beta": cfg.beta,
        "eta": cfg.eta,
        "gamma": cfg.gamma,
        # optional metadata (handy for debugging)
        "_meta": {
            "style": cfg.style,
            "seed": cfg.seed,
            "coords": coords,  # keep coordinates for plotting/debug
        },
    }
    return data

In [4]:
inst = generate_instance(style="clustered", seed=1, n_bases=2, n_techs=6, n_plants=20, n_tickets=40, H=30, X=5)
save_instance_json(inst, "instances/demo_clustered_seed1.json")
print("Saved:", "instances/demo_clustered_seed1.json")

Saved: instances/demo_clustered_seed1.json
